# De los datos al modelo: ¿cuánto debería costar este piso?

**Módulo 10 · Sesión 3 · práctica guiada.** Antes de que tu agente entrene el modelo de tu proyecto, lo haces tú una vez, paso a paso, con los pisos de Madrid. No es tu proyecto todavía: es para entender qué pasa dentro.

[Abrir en Colab](https://colab.research.google.com/github/inesdi-m10-ia-2026-11/curso/blob/main/sesion-03/de-los-datos-al-modelo.ipynb) (si lo estás viendo en GitHub)

**Cómo se usa esto**

- Cada caja gris es código. Se ejecuta con el botón ▶ de su izquierda o con **Mayús + Intro**. Ve en orden, de arriba abajo.
- La primera vez, Colab avisa de que el cuaderno no es de Google: pulsa "Ejecutar de todos modos".
- Donde pone **Tu turno**, cambia el código y vuelve a ejecutar. No se rompe nada: si algo falla, "Entorno de ejecución, Reiniciar sesión y ejecutar todo".
- Para guardar tus cambios: "Archivo, Guardar una copia en Drive".

**Los datos:** 75.455 pisos en venta en Madrid en 2018 (idealista18, de Rey-Blanco, Arbués, López y Páez, 2024; licencia ODbL 1.0). Son precios de oferta de 2018, anonimizados por sus autores.

## 1. Los datos

Una fila es un anuncio. Las columnas son lo que sabemos de cada piso.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/inesdi-m10-ia-2026-11/pisos-madrid/main/data/pisos.csv"
pisos = pd.read_csv(URL)
print(len(pisos), "pisos y", len(pisos.columns), "columnas")
pisos.head()

## 2. Qué queremos saber y con qué

Un modelo aprende de ejemplos **que ya traen la respuesta**. Aquí la respuesta es el `precio`. Lo demás son pistas.

Elegimos 11 pistas que alguien puede rellenar en un formulario. Regla para elegirlas: solo columnas que **conocerás antes** de saber el precio.

In [ ]:
objetivo = "precio"

numeros = ["superficie_m2", "habitaciones", "banos", "latitud", "longitud", "dist_centro_km"]
si_no   = ["ascensor", "terraza", "garaje", "exterior"]
texto   = ["estado"]
columnas = numeros + si_no + texto

pisos[columnas + [objetivo]].describe().round(1)

## 3. Aprender y examinarse

Si le preguntas al modelo por pisos que ya ha visto, acierta siempre y no sabes nada. El examen tiene que ser con pisos **que no ha visto**.

Aquí el corte es por fecha, como en la vida real: aprende con los pisos de enero a septiembre y se examina con los de octubre a diciembre.

In [ ]:
aprender = pisos[pisos["trimestre"] != "2018-T4"]   # enero a septiembre
examen   = pisos[pisos["trimestre"] == "2018-T4"]   # octubre a diciembre

print("Para aprender:", len(aprender), "pisos")
print("Para el examen:", len(examen), "pisos")

## 4. La predicción más tonta

Antes de entrenar nada: ¿cuánto me equivoco si a **todos** los pisos les pongo el precio medio? Ese es el listón. Un modelo que no lo mejora no sirve.

La medida que usamos todo el rato es el **error medio**: de media, cuántos euros me alejo del precio real.

In [ ]:
media = aprender["precio"].mean()
error_media = (examen["precio"] - media).abs().mean()

print(f"Precio medio de los pisos que conozco: {media:,.0f} €")
print(f"Si a todos les pongo ese precio, me equivoco de media en {error_media:,.0f} €")

resultados = {"A todos, la media": error_media}

**Tu turno.** Copia la caja de arriba aquí debajo y cambia `.mean()` por `.median()` (la mediana: el piso que queda en medio). ¿Te equivocas más o menos?

In [ ]:
# Tu turno

## 5. Una línea recta con una sola pista

El modelo más sencillo: "cada metro cuadrado suma tantos euros". Es una **regresión lineal** con una columna.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

recta = LinearRegression()
recta.fit(aprender[["superficie_m2"]], aprender["precio"])          # aquí aprende

prediccion = recta.predict(examen[["superficie_m2"]])               # aquí se examina
error_recta = mean_absolute_error(examen["precio"], prediccion)

print(f"Lo que ha aprendido: cada m² suma {recta.coef_[0]:,.0f} €")
print(f"Error medio en el examen: {error_recta:,.0f} €")
resultados["Recta: solo la superficie"] = error_recta

In [ ]:
import matplotlib.pyplot as plt

muestra = examen.sample(1500, random_state=42)
plt.figure(figsize=(8, 5))
plt.scatter(muestra["superficie_m2"], muestra["precio"], s=6, alpha=0.4, label="pisos del examen")
metros = pd.DataFrame({"superficie_m2": [20, 400]})
plt.plot(metros["superficie_m2"], recta.predict(metros), color="black", linewidth=2, label="lo que dice la recta")
plt.xlabel("Superficie (m²)"); plt.ylabel("Precio (€)"); plt.xlim(0, 400); plt.ylim(0, 2_500_000)
plt.legend(); plt.title("Una recta se queda corta: a igual superficie, los precios cambian mucho")
plt.show()

## 6. Preparar las pistas

Un modelo solo entiende números. Hay que convertir tres cosas:

- **Sí o no** (`ascensor`, `exterior`...): pasan a 1 y 0.
- **Huecos** (hay pisos sin el dato de `exterior`): se rellenan con el valor más habitual.
- **Texto** (`estado`: obra nueva, buen estado, a reformar): una columna de 1 y 0 por cada valor.

Esta receta se guarda **junta con el modelo** (en un *pipeline*), para que un piso nuevo pase por los mismos pasos.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline

def preparar(tabla):
    tabla = tabla[columnas].copy()
    for c in si_no:
        tabla[c] = tabla[c].astype("float")      # True y False pasan a 1 y 0; los huecos siguen vacíos
    return tabla

def receta():
    return ColumnTransformer([
        ("numeros", SimpleImputer(strategy="median"), numeros + si_no),   # rellena huecos
        ("texto", OneHotEncoder(handle_unknown="ignore"), texto),         # texto a columnas de 1 y 0
    ])

X_aprender, y_aprender = preparar(aprender), aprender["precio"]
X_examen,   y_examen   = preparar(examen),   examen["precio"]

def examinar(nombre, modelo):
    modelo.fit(X_aprender, y_aprender)
    error = mean_absolute_error(y_examen, modelo.predict(X_examen))
    resultados[nombre] = error
    print(f"{nombre}: error medio en el examen, {error:,.0f} €")
    return modelo

X_aprender.head(3)

## 7. La misma recta, con las 11 pistas

In [ ]:
lineal = examinar("Recta: las 11 pistas", make_pipeline(receta(), LinearRegression()))

## 8. Un árbol: preguntas en cadena

Un **árbol de decisión** aprende preguntas del tipo "¿tiene más de 104 m²?" y, según la respuesta, hace otra. Al final de cada camino da un precio.

Con solo 3 preguntas se puede leer entero.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text

arbol3 = examinar("Árbol de 3 preguntas", make_pipeline(receta(), DecisionTreeRegressor(max_depth=3, random_state=42)))

nombres = [n.split("__")[1] for n in arbol3[0].get_feature_names_out()]
print()
print(export_text(arbol3[-1], feature_names=nombres, max_depth=3))

## 9. El árbol que se lo aprende de memoria

Si no le pones límite, el árbol hace preguntas hasta separar cada piso. Mira su error con los pisos que ya conoce y con los del examen.

In [ ]:
arbol_libre = examinar("Árbol sin límite", make_pipeline(receta(), DecisionTreeRegressor(random_state=42)))

error_conocidos = mean_absolute_error(y_aprender, arbol_libre.predict(X_aprender))
print(f"Con los pisos que ya había visto: {error_conocidos:,.0f} €")
print("Se los sabe de memoria. Eso se llama sobreajuste: por eso el examen es con pisos nuevos.")

**Tu turno.** Prueba árboles con distinto límite de preguntas: cambia `max_depth` por 6, 10, 14 y 20. ¿Con cuál sale el menor error en el examen?

In [ ]:
# Tu turno: cambia el 6
prueba = make_pipeline(receta(), DecisionTreeRegressor(max_depth=6, random_state=42)).fit(X_aprender, y_aprender)
print(f"Error medio en el examen: {mean_absolute_error(y_examen, prueba.predict(X_examen)):,.0f} €")

## 10. Muchos árboles a la vez

Un árbol solo se equivoca mucho. Muchos árboles que se corrigen entre sí, mucho menos. Hay dos maneras:

- **Bosque aleatorio:** cientos de árboles distintos que votan.
- **Boosting:** árboles pequeños en fila; cada uno corrige los errores del anterior.

El bosque tarda unos segundos.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

bosque   = examinar("Bosque aleatorio", make_pipeline(receta(), RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)))
boosting = examinar("Boosting", make_pipeline(receta(), HistGradientBoostingRegressor(random_state=42)))

El bosque gana por poco. Pero hoy el modelo se va a publicar en un servidor gratuito con 512 MB de memoria. ¿Cuánto pesa cada uno?

In [ ]:
import joblib, os

def peso_mb(modelo):
    joblib.dump(modelo, "prueba.joblib")
    return os.path.getsize("prueba.joblib") / 1_000_000

print(f"Bosque aleatorio: {peso_mb(bosque):,.1f} MB")
print(f"Boosting:         {peso_mb(boosting):,.1f} MB")
print("El mejor en el examen no siempre es el que puedes publicar.")

## 11. La escalera completa

In [ ]:
orden = sorted(resultados.items(), key=lambda par: -par[1])
plt.figure(figsize=(8, 4))
plt.barh([n for n, _ in orden], [e for _, e in orden], color="#00C2CC")
for i, (_, e) in enumerate(orden):
    plt.text(e, i, f"  {e:,.0f} €", va="center")
plt.gca().invert_yaxis(); plt.xlim(0, max(e for _, e in orden) * 1.25)
plt.xlabel("Error medio en el examen (€). Más corto es mejor"); plt.title("Lo que mejora cada paso")
plt.show()

## 12. Usar el modelo

Un piso que no está en los datos: 80 m², 3 habitaciones, 1 baño, exterior, con ascensor, en buen estado, en Chamberí.

In [ ]:
mi_piso = pd.DataFrame([{
    "superficie_m2": 80, "habitaciones": 3, "banos": 1,
    "latitud": 40.4340, "longitud": -3.7038, "dist_centro_km": 2.0,
    "ascensor": True, "terraza": False, "garaje": False, "exterior": True,
    "estado": "buen_estado",        # obra_nueva, buen_estado o a_reformar
}])

estimado = boosting.predict(preparar(mi_piso))[0]
print(f"Precio estimado: {estimado:,.0f} € (precios de oferta de 2018)")

sin_ascensor = mi_piso.assign(ascensor=False)
print(f"El mismo piso sin ascensor: {boosting.predict(preparar(sin_ascensor))[0]:,.0f} €")

**Tu turno.** Cambia los datos por los de un piso que conozcas. La latitud y la longitud salen de Google Maps: clic derecho sobre el portal y el primer renglón del menú son los dos números. `dist_centro_km` es la distancia a la Puerta del Sol.

Y una pregunta: el modelo dice lo que costaba en 2018. ¿Qué habría que hacer para que supiera de hoy?

## 13. La trampa: un resultado demasiado bueno

En los datos hay una columna `precio_m2`. Vamos a dársela al modelo como pista.

In [ ]:
con_trampa = numeros + si_no + ["precio_m2"]

def preparar_trampa(tabla):
    tabla = tabla[con_trampa].copy()
    for c in si_no:
        tabla[c] = tabla[c].astype("float")
    return tabla

tramposo = make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingRegressor(random_state=42))
tramposo.fit(preparar_trampa(aprender), y_aprender)
error_trampa = mean_absolute_error(y_examen, tramposo.predict(preparar_trampa(examen)))
print(f"Error medio en el examen: {error_trampa:,.0f} €")

De decenas de miles de euros a unos pocos miles. No es un modelo mejor: `precio_m2` es el precio dividido entre la superficie, es decir, **la respuesta disfrazada**. Cuando llegue un piso nuevo sin precio, esa columna no existirá.

Esto se llama **fuga de datos**. La señal de alarma es siempre la misma: un resultado demasiado bueno. En tu proyecto, revisa cada pista con esta pregunta: ¿la tendré antes de conocer la respuesta?

## 14. Guardar el modelo

Un modelo entrenado es un archivo. Quien lo cargue puede predecir sin volver a entrenar y sin tener los datos.

In [ ]:
joblib.dump(boosting, "modelo.joblib")
print(f"modelo.joblib: {os.path.getsize('modelo.joblib') / 1_000_000:.1f} MB")

cargado = joblib.load("modelo.joblib")
print(f"El modelo cargado estima: {cargado.predict(preparar(mi_piso))[0]:,.0f} €")

Esto es lo que viene después del descanso: un programa pequeño (una API) carga ese archivo en un servidor y contesta a tu app. Tu app manda los datos de un piso; la API devuelve el precio.

## Lo que te llevas a tu proyecto

Cinco preguntas. Tu agente escribirá el código, pero las respuestas son tuyas y van en `decisiones.md`:

1. ¿Qué columna predigo?
2. ¿Con qué pistas? Solo las que conoceré antes de saber la respuesta.
3. ¿Con qué datos se examina el modelo? Unos que no haya visto.
4. ¿Cuál es mi predicción tonta y cuánto se equivoca?
5. ¿Cuánto se equivoca mi modelo, en unidades que yo entienda? ¿Me vale?

---

# Si vas por delante

Seis extras, de menos a más difícil. No hace falta hacerlos todos ni en orden.

## Extra A. ¿Qué pistas pesan más?

Se baraja una columna (se le quita el sentido) y se mira cuánto empeora el modelo. Cuanto más empeora, más pesaba esa pista.

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(boosting, X_examen, y_examen, n_repeats=3, random_state=42, scoring="neg_mean_absolute_error")
peso = pd.Series(imp.importances_mean, index=columnas).sort_values()

plt.figure(figsize=(8, 4))
plt.barh(peso.index, peso.values, color="#00C2CC")
plt.xlabel("Cuánto sube el error medio (€) al barajar la columna"); plt.title("Qué pistas pesan más")
plt.show()

¿Te sorprende que `habitaciones` pese tan poco? Pista: mira qué otra columna cuenta casi lo mismo.

## Extra B. ¿Dónde falla el modelo?

Un error medio esconde mucho. Míralo por distrito y por tramo de precio.

In [ ]:
revision = examen.assign(estimado=boosting.predict(X_examen))
revision["error"] = (revision["estimado"] - revision["precio"]).abs()
revision["error_pct"] = 100 * revision["error"] / revision["precio"]

por_distrito = revision.groupby("distrito").agg(pisos=("error", "size"), error_medio=("error", "mean"), error_pct_tipico=("error_pct", "median"))
print(por_distrito.sort_values("error_pct_tipico").round(1))

tramos = pd.cut(revision["precio"], [0, 150_000, 300_000, 600_000, 1_000_000, 4_000_000])
print(revision.groupby(tramos, observed=True).agg(pisos=("error", "size"), error_medio=("error", "mean"), error_pct_tipico=("error_pct", "median")).round(1))

En euros falla mucho más en los pisos caros; en porcentaje, casi igual en todos. ¿Con qué medida te quedarías para explicar tu modelo a alguien que no es técnico?

## Extra C. Cuando la respuesta no es un número

Predecir un número es **regresión**. Predecir una etiqueta (sí o no, tipo A o B) es **clasificación**. Cambia la medida: ya no es un error en euros, es el **porcentaje de aciertos**. Y cambia la predicción tonta: decir siempre la etiqueta más frecuente.

Aquí la etiqueta: ¿este piso es caro para su distrito? (su precio por m² está por encima de la mediana del distrito).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

mediana_distrito = aprender.groupby("distrito")["precio_m2"].median()
caro_aprender = (aprender["precio_m2"] > aprender["distrito"].map(mediana_distrito)).astype(int)
caro_examen   = (examen["precio_m2"]   > examen["distrito"].map(mediana_distrito)).astype(int)

clasificador = make_pipeline(receta(), HistGradientBoostingClassifier(random_state=42)).fit(X_aprender, caro_aprender)
aciertos = accuracy_score(caro_examen, clasificador.predict(X_examen))
tonta = max(caro_examen.mean(), 1 - caro_examen.mean())

print(f"Aciertos del modelo: {100 * aciertos:.1f} %")
print(f"Diciendo siempre la etiqueta más frecuente: {100 * tonta:.1f} %")

## Extra D. ¿Cuántos datos hacen falta?

El mismo modelo, aprendiendo con menos pisos.

In [ ]:
for n in [500, 2000, 5000, 20000, len(aprender)]:
    trozo = aprender if n == len(aprender) else aprender.sample(n, random_state=42)
    m = make_pipeline(receta(), HistGradientBoostingRegressor(random_state=42)).fit(preparar(trozo), trozo["precio"])
    print(f"{n:>6} pisos: error medio en el examen, {mean_absolute_error(y_examen, m.predict(X_examen)):,.0f} €")

A partir de cierto punto, más datos casi no ayudan. ¿Cuántas filas tiene tu proyecto?

## Extra E. Un rango en vez de un número

"Vale 400.000 €" suena más seguro de lo que es. Un modelo puede dar un rango: "entre tanto y tanto, 8 de cada 10 veces".

In [ ]:
bajo = make_pipeline(receta(), HistGradientBoostingRegressor(loss="quantile", quantile=0.1, random_state=42)).fit(X_aprender, y_aprender)
alto = make_pipeline(receta(), HistGradientBoostingRegressor(loss="quantile", quantile=0.9, random_state=42)).fit(X_aprender, y_aprender)

print(f"Mi piso: entre {bajo.predict(preparar(mi_piso))[0]:,.0f} € y {alto.predict(preparar(mi_piso))[0]:,.0f} €")

dentro = ((y_examen >= bajo.predict(X_examen)) & (y_examen <= alto.predict(X_examen))).mean()
print(f"En el examen, el precio real cae dentro del rango el {100 * dentro:.0f} % de las veces")

## Extra F. Con tus datos

Sube el CSV de tu proyecto y repite los pasos 2 a 10 con tus columnas. Es la mejor preparación para la segunda parte de la sesión.

Si tus datos son de tu empresa o tienen datos personales, **no los subas**: usa una muestra con los valores cambiados.

In [ ]:
# Quita los # de las tres líneas siguientes y ejecuta. Solo funciona en Colab.
# from google.colab import files
# subido = files.upload()
# mios = pd.read_csv(next(iter(subido)))

# Después: mios.head(), elige tu objetivo y tus pistas, y sigue desde el paso 3.